In [11]:

from langgraph.graph import StateGraph, START,END, add_messages
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage,HumanMessage
import operator
from langgraph.checkpoint.memory import MemorySaver


In [2]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="qwen2.5-coder:7b", base_url="http://localhost:11434")


c:\numerix-python-project\ai-code-assistance\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


In [4]:
def chat_node(state: ChatState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


In [12]:
thread_id = 1
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)
checkpointers = MemorySaver()

graph = graph.compile(checkpointer=checkpointers)
config = {"configurable": {"thread_id": thread_id}}

result = graph.invoke({"messages": [HumanMessage(content="Tell me a joke")]},config=config)
print(result["messages"][-1].content)

Why did the tomato turn red?

Because it saw the salad dressing!


In [14]:
while True:
  
    user_input = input("Type Here: ")
    print(f'Your Question : {user_input}')
    if user_input.lower() in ["quit", "exit", "bye"]:
        break

    response = graph.invoke({"messages": [HumanMessage(content=user_input)]},config)
    print("AI:", response["messages"][-1].content)


Your Question : Hi My name is Rajesh
AI: Hello Rajesh! How can I assist you today?
Your Question : What is my Name
AI: Your name is Rajesh.
Your Question : Can you add 100 and 10
AI: 100 + 10 equals 110.
Your Question : Can you multiply 2 to result 
AI: 2 multiplied by any number will give you that number doubled. For example, if you want to double 5, the result would be 10. Please specify a number for me to provide the exact result of multiplying it by 2.
Your Question : bye


In [15]:
graph.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='Tell me a joke', additional_kwargs={}, response_metadata={}, id='ef870bbd-26eb-470c-a684-2fc5ad27aa15'), AIMessage(content='Why did the tomato turn red?\n\nBecause it saw the salad dressing!', additional_kwargs={}, response_metadata={'model': 'qwen2.5-coder:7b', 'created_at': '2026-05-09T15:20:35.336364552Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9161811397, 'load_duration': 6153270412, 'prompt_eval_count': 33, 'prompt_eval_duration': 1176414002, 'eval_count': 15, 'eval_duration': 1809922791, 'logprobs': None, 'model_name': 'qwen2.5-coder:7b', 'model_provider': 'ollama'}, id='lc_run--019e0d53-5710-7e70-9a81-7e77a455aa86-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 33, 'output_tokens': 15, 'total_tokens': 48}), HumanMessage(content='Hi My name is Rajesh', additional_kwargs={}, response_metadata={}, id='b0724538-6ac7-4a6a-88fa-0216637c71a2'), AIMessage(content='Hello Rajesh! How can I a